In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

# Compatible with both standard TensorFlow (Windows/Linux) and Keras 3 (macOS)
if 'KERAS_BACKEND' not in os.environ:
    os.environ['KERAS_BACKEND'] = 'torch'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    import keras
    from keras.models import Sequential
    from keras.layers import Dense, Input
except ImportError:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Input

print('Libraries imported successfully.')


Libraries imported successfully.


In [2]:
df = pd.read_csv('BostonHousing.csv')
print(f"Dataset Shape: {df.shape}")
df.head()


Dataset Shape: (506, 14)
      crim    zn  indus  chas    nox     rm   age     dis  rad  tax  ptratio       b  lstat  medv
0  0.00632  18.0   2.31     0  0.538  6.575  65.2  4.0900    1  296     15.3  396.90   4.98  24.0
1  0.02731   0.0   7.07     0  0.469  6.421  78.9  4.9671    2  242     17.8  396.90   9.14  21.6
2  0.02729   0.0   7.07     0  0.469  7.185  61.1  4.9671    2  242     17.8  392.83   4.03  34.7
3  0.03237   0.0   2.18     0  0.458  6.998  45.8  6.0622    3  222     18.7  394.63   2.94  33.4
4  0.06905   0.0   2.18     0  0.458  7.147  54.2  6.0622    3  222     18.7  396.90   5.33  36.2



In [3]:
X = df.drop(columns=['medv']).values
y = df['medv'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Samples: {len(X_train)}, Testing Samples: {len(X_test)}")


Training Samples: 404, Testing Samples: 102


In [4]:
model = Sequential([
    Input(shape=(13,)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])
model.summary()


Model: "sequential"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 3,009 (11.75 KB)
 Trainable params: 3,009 (11.75 KB)
 Non-trainable params: 0 (0.00 B)



In [5]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = model.fit(X_train_scaled, y_train, epochs=60, batch_size=16, verbose=0)
print(f"Training finished. Final Loss (MSE): {history.history['loss'][-1]:.4f}, MAE: {history.history['mae'][-1]:.4f}")


Training finished. Final Loss (MSE): 7.8058, MAE: 2.0146


In [6]:
y_pred = model.predict(X_test_scaled, verbose=0).flatten()

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R2 Score: {r2:.4f}")


Mean Squared Error (MSE): 12.14
Root Mean Squared Error (RMSE): 3.48
Mean Absolute Error (MAE): 2.35
R2 Score: 0.8344



In [7]:
print("Sample Predictions (Actual vs Predicted in s):")
for i in range(5):
    diff = abs(y_test[i] - y_pred[i])
    print(f"Sample {i+1}: Actual = {y_test[i]:.2f}, Predicted = {y_pred[i]:.2f}, Difference = {diff:.2f}")


Sample Predictions (Actual vs Predicted in s):
Sample 1: Actual = 23.60, Predicted = 27.48, Difference = 3.88
Sample 2: Actual = 32.40, Predicted = 33.27, Difference = 0.87
Sample 3: Actual = 13.60, Predicted = 17.71, Difference = 4.11
Sample 4: Actual = 22.80, Predicted = 25.43, Difference = 2.63
Sample 5: Actual = 16.10, Predicted = 16.75, Difference = 0.65

